# Deep Learning Lab — Fashion-MNIST with a Multilayer Perceptron

## From 28×28 pixels to a 10-class prediction

In this lab, a multilayer perceptron (MLP) will classify grayscale clothing images from **Fashion-MNIST**.

The complete flow is:

**image → normalize → flatten → hidden layers → class probabilities → predicted class**

The main objective is not only to obtain an accuracy value. The code should make every stage of the neural network understandable:

1. what enters the network;
2. what each Dense layer computes;
3. why nonlinear activation is required;
4. why the output layer has 10 neurons;
5. how Softmax converts outputs into probabilities;
6. how the loss guides training;
7. how training and validation curves are interpreted;
8. how predictions and errors are inspected.

## Learning outcomes

By the end of the lab, you should be able to:

- describe the Fashion-MNIST input and output structure;
- explain why a $28\times28$ image becomes a vector of 784 values for an MLP;
- build a Keras MLP with configurable hidden layers;
- explain the role of Dense, ReLU, Dropout, Softmax, Adam, and cross-entropy;
- train the network using training and validation data;
- interpret accuracy and loss curves;
- evaluate the final model on unseen test data;
- inspect class probabilities, correct predictions, and mistakes;
- compare two MLP architectures and justify which one generalizes better.

## Quick glossary

| Term | Meaning |
|---|---|
| **Sample** | One image and its label. |
| **Feature** | One numeric input value. In the flattened image, each pixel becomes one feature. |
| **Label** | Correct class index from 0 to 9. |
| **Batch** | A small group of samples processed before one parameter update. |
| **Epoch** | One complete pass through the training set. |
| **Dense layer** | Every neuron receives input from every output of the previous layer. |
| **Weight** | Trainable value that controls the influence of an input. |
| **Bias** | Trainable offset added before activation. |
| **Activation** | Nonlinear transformation applied after the weighted sum. |
| **ReLU** | $\operatorname{ReLU}(z)=\max(0,z)$. |
| **Logit** | Raw output value before Softmax. |
| **Softmax** | Converts the final 10 logits into probabilities that sum to 1. |
| **Loss** | Numeric training objective that the optimizer attempts to reduce. |
| **Optimizer** | Algorithm that updates the trainable weights and biases. |
| **Learning rate** | Size of an optimizer update step. |
| **Dropout** | During training, randomly disables a fraction of activations to reduce over-reliance on particular neurons. |
| **Training set** | Data used to update model parameters. |
| **Validation set** | Data used to monitor generalization during training. |
| **Test set** | Unseen data used for final evaluation. |
| **Underfitting** | Model is too limited to learn the useful pattern. |
| **Overfitting** | Model fits training data much better than unseen data. |

## Code map

The notebook is organized so that each code block answers one question:

| Question | Code stage |
|---|---|
| What does the dataset contain? | load and visualize |
| How does an image become MLP input? | normalize and flatten |
| What transformation does each layer perform? | Dense + ReLU |
| How is the network assembled? | `build_mlp()` |
| How are parameters learned? | `model.fit()` |
| Is learning generalizing? | accuracy and loss curves |
| How well does the model perform on unseen data? | test evaluation |
| What is the model predicting? | Softmax probabilities |
| Which classes are confused? | confusion matrix |
| Does architecture size matter? | required comparison |

In [ ]:
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Random seed:", SEED)

In [ ]:
CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

NUM_CLASSES = len(CLASS_NAMES)
IMAGE_HEIGHT = 28
IMAGE_WIDTH = 28
FLAT_FEATURES = IMAGE_HEIGHT * IMAGE_WIDTH

print("Classes:", NUM_CLASSES)
print("Pixels per image:", FLAT_FEATURES)

In [ ]:
# Keras provides Fashion-MNIST directly.
# The first call may download the dataset if it is not already cached.
(train_images_raw, train_labels_raw), (test_images_raw, test_labels) = (
    keras.datasets.fashion_mnist.load_data()
)

print("Raw training images:", train_images_raw.shape)
print("Raw training labels:", train_labels_raw.shape)
print("Raw test images:", test_images_raw.shape)
print("Raw test labels:", test_labels.shape)
print("Pixel-value range:", train_images_raw.min(), "to", train_images_raw.max())

# Stage 1 — Inspect Fashion-MNIST

Each sample is a $28\times28$ grayscale image.

Labels are integer class IDs from 0 to 9.

In [ ]:
def show_image_grid(images, labels, class_names, count=12):
    count = min(count, len(images))

    cols = 4
    rows = int(np.ceil(count / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(10, 2.5 * rows))
    axes = np.asarray(axes).reshape(-1)

    for i in range(count):
        axes[i].imshow(images[i], cmap="gray")
        axes[i].set_title(f"{labels[i]} · {class_names[int(labels[i])]}")
        axes[i].axis("off")

    for i in range(count, len(axes)):
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


show_image_grid(
    train_images_raw,
    train_labels_raw,
    CLASS_NAMES,
    count=12,
)

The raw images contain integer pixel intensities between 0 and 255.

For an MLP, two preprocessing steps are required:

1. normalize the pixel values;
2. flatten the two-dimensional image into one vector.

# Stage 2 — Normalize and flatten

Normalization:

$$
x_{\text{normalized}}
=
\frac{x}{255}
$$

Flattening:

$$
28\times28
=
784
$$

so

$$
(28,28)
\longrightarrow
(784,)
$$

In [ ]:
# Convert integer pixel values to floating point and scale them to [0, 1].
train_images = (
    train_images_raw.astype("float32") / 255.0
).reshape(-1, FLAT_FEATURES)

test_images = (
    test_images_raw.astype("float32") / 255.0
).reshape(-1, FLAT_FEATURES)

print("Flattened training images:", train_images.shape)
print("Flattened test images:", test_images.shape)
print("Normalized range:", train_images.min(), "to", train_images.max())

In [ ]:
def make_fast_lab_split(
    images,
    labels,
    train_size=12000,
    val_size=2000,
    seed=2026,
):
    """
    Shuffle the original training data and create a smaller split
    so the lab trains quickly on a CPU.
    """
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(images))

    train_idx = indices[:train_size]
    val_idx = indices[train_size:train_size + val_size]

    return (
        images[train_idx],
        labels[train_idx],
        images[val_idx],
        labels[val_idx],
    )

In [ ]:
train_x, train_y, val_x, val_y = make_fast_lab_split(
    train_images,
    train_labels_raw,
    train_size=12000,
    val_size=2000,
    seed=SEED,
)

print("Training subset:", train_x.shape, train_y.shape)
print("Validation subset:", val_x.shape, val_y.shape)
print("Test set:", test_images.shape, test_labels.shape)

## What one Dense layer computes

For layer $\ell$,

$$
\mathbf{z}^{(\ell)}
=
\mathbf{W}^{(\ell)}
\mathbf{a}^{(\ell-1)}
+
\mathbf{b}^{(\ell)}
$$

Then an activation function is applied:

$$
\mathbf{a}^{(\ell)}
=
g\left(\mathbf{z}^{(\ell)}\right)
$$

### Symbols

- $\mathbf{a}^{(\ell-1)}$: input vector to the current layer;
- $\mathbf{W}^{(\ell)}$: trainable weight matrix;
- $\mathbf{b}^{(\ell)}$: trainable bias vector;
- $\mathbf{z}^{(\ell)}$: weighted sum before activation;
- $g(\cdot)$: activation function;
- $\mathbf{a}^{(\ell)}$: output of the layer.

For the hidden layers, ReLU is used:

$$
\operatorname{ReLU}(z)=\max(0,z)
$$

ReLU allows the network to build nonlinear decision boundaries. Without nonlinear activations, stacking Dense layers would still reduce to one linear transformation.

## Why the output layer has 10 neurons

Fashion-MNIST contains 10 mutually exclusive classes.

The output layer therefore contains one neuron per class:

$$
[z_0,z_1,\ldots,z_9]
$$

Softmax converts those logits into probabilities:

$$
p_k
=
\frac{e^{z_k}}
{\displaystyle\sum_{j=0}^{9}e^{z_j}}
$$

with

$$
\sum_{k=0}^{9}p_k=1
$$

The predicted class is

$$
\hat{y}
=
\operatorname*{arg\,max}_{k} p_k
$$

Example:

$$
[0.01,\ 0.02,\ 0.03,\ 0.04,\ 0.05,\ 0.08,\ 0.10,\ 0.12,\ 0.50,\ 0.05]
$$

The largest probability is at index 8, so the predicted class is class 8, **Bag**.

In [ ]:
def draw_mlp_diagram(hidden_layers=(256, 128), num_classes=10):
    """
    Draw a simple block-level view of the MLP.
    This deliberately avoids drawing hundreds of individual neurons.
    """
    labels = [
        "Input\n784 pixels",
        *[f"Hidden\n{units} ReLU" for units in hidden_layers],
        f"Output\n{num_classes} Softmax",
    ]

    x_positions = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(11, 3.8))

    for i, (x, label) in enumerate(zip(x_positions, labels)):
        ax.scatter([x], [0], s=4200, edgecolor="black", linewidth=1.5, zorder=3)
        ax.text(x, 0, label, ha="center", va="center", fontsize=11, weight="bold")

        if i < len(labels) - 1:
            ax.annotate(
                "",
                xy=(x_positions[i + 1] - 0.22, 0),
                xytext=(x + 0.22, 0),
                arrowprops=dict(arrowstyle="->", lw=2),
            )

    ax.set_xlim(-0.7, len(labels) - 0.3)
    ax.set_ylim(-1.0, 1.0)
    ax.set_title("Multilayer Perceptron used in this lab")
    ax.axis("off")
    plt.show()


draw_mlp_diagram((256, 128), NUM_CLASSES)

## How many parameters are being trained?

A Dense layer with

- $n_{\text{in}}$ input values, and
- $n_{\text{out}}$ neurons

contains

$$
(n_{\text{in}}+1)n_{\text{out}}
$$

trainable parameters.

The extra $1$ represents one bias per output neuron.

For the baseline network:

### Input 784 → hidden 256

$$
(784+1)\times256
=
200{,}960
$$

### Hidden 256 → hidden 128

$$
(256+1)\times128
=
32{,}896
$$

### Hidden 128 → output 10

$$
(128+1)\times10
=
1{,}290
$$

Total:

$$
200{,}960+32{,}896+1{,}290
=
235{,}146
$$

Dropout has no trainable parameters.

# Stage 3 — Build a configurable MLP

The function below is dynamic: the list `hidden_layers` determines both the number of hidden layers and the number of neurons in each layer.

For example:

```python
hidden_layers=(256, 128)
```

creates two hidden layers.

```python
hidden_layers=(64,)
```

creates one hidden layer.

In [ ]:
def build_mlp(
    hidden_layers=(256, 128),
    activation="relu",
    dropout_rate=0.20,
    learning_rate=1e-3,
):
    """
    Build and compile a fully connected neural network.

    hidden_layers:
        Tuple containing the number of neurons in every hidden layer.

    activation:
        Hidden-layer activation function.

    dropout_rate:
        Fraction of hidden activations randomly disabled during training.

    learning_rate:
        Adam optimizer step size.
    """
    model = keras.Sequential(name="fashion_mnist_mlp")

    # The MLP receives one vector containing 784 normalized pixels.
    model.add(
        layers.Input(
            shape=(FLAT_FEATURES,),
            name="pixels",
        )
    )

    # Build as many hidden layers as requested.
    for layer_index, units in enumerate(hidden_layers, start=1):
        model.add(
            layers.Dense(
                units,
                activation=activation,
                name=f"hidden_{layer_index}",
            )
        )

        # Dropout is active during training only.
        if dropout_rate > 0:
            model.add(
                layers.Dropout(
                    dropout_rate,
                    name=f"dropout_{layer_index}",
                )
            )

    # One output neuron for each Fashion-MNIST class.
    # Softmax converts the 10 outputs into probabilities.
    model.add(
        layers.Dense(
            NUM_CLASSES,
            activation="softmax",
            name="class_probabilities",
        )
    )

    # Adam updates the trainable parameters.
    # Sparse categorical cross-entropy matches integer class labels.
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


model = build_mlp(
    hidden_layers=(256, 128),
    activation="relu",
    dropout_rate=0.20,
    learning_rate=1e-3,
)

model.summary()

## Why `sparse_categorical_crossentropy` is used

The labels are stored as integer class indices such as

```text
9
0
3
...
```

rather than one-hot vectors.

For one sample with correct class $y$, the loss is

$$
\mathcal{L}
=
-\log(p_y)
$$

For $N$ samples,

$$
\mathcal{L}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\log\left(p_{i,y_i}\right)
$$

A confident correct prediction gives a small loss.  
A confident incorrect prediction gives a large loss.

`SparseCategoricalCrossentropy` is therefore appropriate when the target is one integer class index and the model outputs a probability distribution over multiple classes.

# Stage 4 — Train the baseline model

One epoch means one complete pass through the training subset.

A batch size of 128 means the model processes 128 training samples before the optimizer performs the next update.

In [ ]:
EPOCHS = 8
BATCH_SIZE = 128

start_time = time.perf_counter()

history = model.fit(
    train_x,
    train_y,
    validation_data=(val_x, val_y),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)

baseline_training_time = time.perf_counter() - start_time

print(f"Training time: {baseline_training_time:.2f} seconds")

In [ ]:
def plot_training_history(history):
    epochs = range(1, len(history.history["accuracy"]) + 1)

    plt.figure(figsize=(8, 4.5))
    plt.plot(epochs, history.history["accuracy"], marker="o", label="training")
    plt.plot(epochs, history.history["val_accuracy"], marker="o", label="validation")
    plt.title("Accuracy across epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 4.5))
    plt.plot(epochs, history.history["loss"], marker="o", label="training")
    plt.plot(epochs, history.history["val_loss"], marker="o", label="validation")
    plt.title("Loss across epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

In [ ]:
plot_training_history(history)

## Reading the curves

A useful model should improve on both training and validation data.

### Underfitting

Training and validation performance both remain weak.

### Overfitting

Training performance keeps improving while validation improvement stops or reverses.

The validation curve is therefore not decoration. It is evidence about generalization.

# Stage 5 — Final evaluation on the test set

The test set was not used to update model parameters and was not used for validation during training.

It provides the final unseen-data measurement.

In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_images,
    test_labels,
    verbose=0,
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

# Stage 6 — Inspect Softmax probabilities

The network returns one probability for each of the 10 classes.

`np.argmax(..., axis=1)` selects the class with the largest probability for every image.

In [ ]:
test_probabilities = model.predict(
    test_images,
    verbose=0,
)

predicted_labels = np.argmax(
    test_probabilities,
    axis=1,
)

print("Probability tensor shape:", test_probabilities.shape)
print("First probability vector:")
print(np.round(test_probabilities[0], 3))
print("Sum:", round(float(test_probabilities[0].sum()), 6))
print("Predicted class:", predicted_labels[0], CLASS_NAMES[predicted_labels[0]])
print("True class:", test_labels[0], CLASS_NAMES[test_labels[0]])

In [ ]:
def show_predictions(images_2d, true_labels, probabilities, class_names, count=12):
    predicted_labels = np.argmax(probabilities, axis=1)
    confidences = np.max(probabilities, axis=1)

    count = min(count, len(images_2d))
    cols = 4
    rows = int(np.ceil(count / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(11, 2.8 * rows))
    axes = np.asarray(axes).reshape(-1)

    for i in range(count):
        true_id = int(true_labels[i])
        pred_id = int(predicted_labels[i])

        axes[i].imshow(images_2d[i], cmap="gray")
        axes[i].set_title(
            f"True: {class_names[true_id]}\n"
            f"Pred: {class_names[pred_id]} ({confidences[i]:.2f})"
        )
        axes[i].axis("off")

    for i in range(count, len(axes)):
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


def show_misclassified_examples(
    images_2d,
    true_labels,
    probabilities,
    class_names,
    count=12,
):
    predicted_labels = np.argmax(probabilities, axis=1)
    wrong_indices = np.where(predicted_labels != true_labels)[0]

    if len(wrong_indices) == 0:
        print("No misclassified samples found.")
        return

    selected = wrong_indices[:count]

    show_predictions(
        images_2d[selected],
        true_labels[selected],
        probabilities[selected],
        class_names,
        count=len(selected),
    )

In [ ]:
show_predictions(
    test_images_raw,
    test_labels,
    test_probabilities,
    CLASS_NAMES,
    count=12,
)

# Stage 7 — Confusion matrix

Accuracy compresses the entire test set into one number.

The confusion matrix shows where the mistakes occur.

In [ ]:
def plot_confusion_matrix(true_labels, predicted_labels, class_names):
    cm = tf.math.confusion_matrix(
        true_labels,
        predicted_labels,
        num_classes=len(class_names),
    ).numpy()

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(cm)

    ax.set_title("Confusion matrix on the test set")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    threshold = cm.max() / 2

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if cm[i, j] > 0:
                ax.text(
                    j,
                    i,
                    cm[i, j],
                    ha="center",
                    va="center",
                    fontsize=8,
                )

    plt.colorbar(im, ax=ax, label="Number of samples")
    plt.tight_layout()
    plt.show()

    return cm

In [ ]:
cm = plot_confusion_matrix(
    test_labels,
    predicted_labels,
    CLASS_NAMES,
)

# Stage 8 — Inspect misclassified images

A useful model analysis includes examples the model gets wrong.

Look for visually similar classes and low-confidence decisions.

In [ ]:
show_misclassified_examples(
    test_images_raw,
    test_labels,
    test_probabilities,
    CLASS_NAMES,
    count=12,
)

# Stage 9 — Required architecture experiment

The baseline is compared with a much smaller MLP.

Baseline:

```text
784 → 256 → 128 → 10
```

Smaller model:

```text
784 → 64 → 10
```

The comparison uses the same data, batch size, and epoch count.

In [ ]:
small_model = build_mlp(
    hidden_layers=(64,),
    activation="relu",
    dropout_rate=0.0,
    learning_rate=1e-3,
)

small_model.summary()

start_time = time.perf_counter()

small_history = small_model.fit(
    train_x,
    train_y,
    validation_data=(val_x, val_y),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)

small_training_time = time.perf_counter() - start_time

small_test_loss, small_test_accuracy = small_model.evaluate(
    test_images,
    test_labels,
    verbose=0,
)

print(f"Small-model training time: {small_training_time:.2f} seconds")
print(f"Small-model test loss: {small_test_loss:.4f}")
print(f"Small-model test accuracy: {small_test_accuracy:.4f}")

In [ ]:
comparison = {
    "baseline": {
        "final_train_accuracy": history.history["accuracy"][-1],
        "final_val_accuracy": history.history["val_accuracy"][-1],
        "final_val_loss": history.history["val_loss"][-1],
        "test_accuracy": test_accuracy,
        "training_time_sec": baseline_training_time,
        "parameters": model.count_params(),
    },
    "small": {
        "final_train_accuracy": small_history.history["accuracy"][-1],
        "final_val_accuracy": small_history.history["val_accuracy"][-1],
        "final_val_loss": small_history.history["val_loss"][-1],
        "test_accuracy": small_test_accuracy,
        "training_time_sec": small_training_time,
        "parameters": small_model.count_params(),
    },
}

for model_name, values in comparison.items():
    print(f"\n{model_name.upper()}")
    for key, value in values.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(
    range(1, EPOCHS + 1),
    history.history["val_accuracy"],
    marker="o",
    label="baseline 256-128",
)
plt.plot(
    range(1, EPOCHS + 1),
    small_history.history["val_accuracy"],
    marker="o",
    label="small 64",
)
plt.title("Validation accuracy: architecture comparison")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.legend()
plt.show()

## How to interpret the architecture comparison

Do not conclude that a larger network is automatically better.

Use evidence:

- final training accuracy;
- final validation accuracy;
- validation loss;
- test accuracy;
- parameter count;
- training time.

If both training and validation results are weak, the smaller model may be underfitting.

If training performance is much stronger than validation performance, excessive capacity may be contributing to overfitting.

# Stage 10 — Activation functions

The baseline uses ReLU.

The curves below show how several common activations transform the same input value.

In [ ]:
z = np.linspace(-5, 5, 400)

activation_values = {
    "ReLU": tf.nn.relu(z).numpy(),
    "Tanh": tf.nn.tanh(z).numpy(),
    "Sigmoid": tf.nn.sigmoid(z).numpy(),
    "ELU": tf.nn.elu(z).numpy(),
}

plt.figure(figsize=(8, 5))
for name, values in activation_values.items():
    plt.plot(z, values, label=name)

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.title("Common activation functions")
plt.xlabel("Input z")
plt.ylabel("Activation output")
plt.legend()
plt.show()

## Final interpretation checklist

A complete lab explanation should answer:

1. Why is each image flattened from $28\times28$ to 784 values?
2. What does one Dense neuron compute?
3. Why is ReLU placed in hidden layers?
4. Why does the output layer contain 10 neurons?
5. What does Softmax mean?
6. Why is sparse categorical cross-entropy used?
7. What does one epoch mean?
8. What does batch size mean?
9. How can the curves reveal underfitting or overfitting?
10. Which classes are confused most often?
11. How did the smaller MLP differ from the baseline?
12. What is one limitation of flattening images for an MLP?

### Final lab message

**An MLP learns a sequence of nonlinear transformations that convert a vector of pixels into a probability distribution over classes.**